In [1]:
# Clone your GitHub repo
!git clone https://github.com/Kashshaf-Labib/synth-dataset-pipeline.git
%cd synth-dataset-pipeline

Cloning into 'synth-dataset-pipeline'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 88 (delta 53), reused 69 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 644.59 KiB | 12.64 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/kaggle/working/synth-dataset-pipeline


In [3]:
# Install dependencies
!pip install -q langchain langchain-google-genai langchain-google-vertexai \
    langchain-openai openai google-generativeai google-cloud-aiplatform \
    google-genai pandas python-dotenv Pillow pydantic tqdm

In [4]:
# Tell the Google SDKs which project to bill to
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "spai-img-gen" # Your GCP Project ID
print("✅ Kaggle Google Cloud integration active!")


✅ Kaggle Google Cloud integration active!


In [5]:
import os
from kaggle_secrets import UserSecretsClient

# Load API key from Kaggle Secrets (never hardcode!)
secrets = UserSecretsClient()
google_api_key = secrets.get_secret("GOOGLE_API_KEY")

env_content = f"""
# === API Keys ===
OPENAI_API_KEY=
GOOGLE_API_KEY={google_api_key}

# === Vertex AI Config ===
GCP_PROJECT_ID=spai-img-gen
GCP_LOCATION=global
GCP_IMAGE_LOCATION=global

# === Pipeline Config ===
IMAGE_GENERATOR=gemini
CHAT_MODEL=gemini-3-flash-preview
GEMINI_IMAGE_MODEL=gemini-3-pro-image-preview
DALLE_MODEL=dall-e-3
DALLE_IMAGE_SIZE=1024x1024
DALLE_IMAGE_QUALITY=standard
""".strip()

with open(".env", "w") as f:
    f.write(env_content)
print("✅ .env created (API key loaded from Kaggle Secrets)")


✅ .env created (API key loaded from Kaggle Secrets)


In [6]:
import os

images_dir = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized"
sample_files = os.listdir(images_dir)[:5]
print(f"✅ Images directory has {len(os.listdir(images_dir))} files")
print(f"   Sample: {sample_files}")

✅ Images directory has 7470 files
   Sample: ['349_IM-1697-2001.dcm.png', '607_IM-2196-1001.dcm.png', '2832_IM-1249-2001.dcm.png', '699_IM-2263-2001.dcm.png', '1931_IM-0602-2001.dcm.png']


In [7]:
from pipeline.report_parser import load_projections, load_reports

proj = load_projections("indiana_projections.csv")
print(f"✅ {len(proj)} uids with projection data")
print(f"   uid=1: {[(i.filename, i.projection) for i in proj[1]]}")

# Check images actually exist on disk
uid1_paths = [f"{images_dir}/{i.filename}" for i in proj[1]]
for p in uid1_paths:
    print(f"   {'✅' if os.path.exists(p) else '❌'} {p}")

✅ 3851 uids with projection data
   uid=1: [('1_IM-0001-4001.dcm.png', 'Frontal'), ('1_IM-0001-3001.dcm.png', 'Lateral')]
   ✅ /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/1_IM-0001-4001.dcm.png
   ✅ /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/1_IM-0001-3001.dcm.png


In [8]:
!python -m pipeline \
    --csv indiana_reports.csv \
    --projections-csv indiana_projections.csv \
    --images-dir /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized \
    --limit 3 \
    --generator gemini

🗂  Loading projections from indiana_projections.csv ...
   → 3851 uids with projection data

📂 Loading reports from indiana_reports.csv ...
   → 3 reports loaded

🤖 Initializing LLM chain (model: gemini-3-flash-preview) ...
   → Chain ready

  📐 uid=1 → 2 views detected: ['PA', 'Lateral']                                
  📎 uid=1 → 2 reference image(s) (Frontal, Lateral)                            
Processing reports:   0%|                             | 0/3 [00:07<?, ?report/s]  ✓ Image saved: 1_PA.png (model: gemini-3-pro-image-preview, 2 reference image(s))
  ✓ uid=1 [PA] → 1_PA.png                                                       
Processing reports:   0%|                             | 0/3 [00:28<?, ?report/s]  ✓ Image saved: 1_Lateral.png (model: gemini-3-pro-image-preview, 2 reference image(s))
  ✓ uid=1 [Lateral] → 1_Lateral.png                                             
  📐 uid=2 → 2 views detected: ['Frontal', 'Lateral']                           
  📎 uid=2 → 2 reference

In [22]:
from PIL import Image
import matplotlib.pyplot as plt
import json

# Show metadata
with open("output/metadata.json") as f:
    meta = json.load(f)
for entry in meta:
    print(f"uid={entry['uid']}")
    if 'reference_images' in entry:
        print(f"  Reference images: {entry['reference_images']}")
    if 'image_path' in entry:
        print(f"  Generated image: {entry['image_path']}")
        img = Image.open(entry['image_path'])
        plt.figure(figsize=(6, 6))
        plt.imshow(img, cmap='gray')
        plt.title(f"Synthetic X-ray (uid={entry['uid']})")
        plt.axis('off')
        plt.show()
    elif 'error_image' in entry:
        print(f"  ❌ Error: {entry['error_image']}")


uid=1
  Reference images: [{'filename': '1_IM-0001-4001.dcm.png', 'projection': 'Frontal'}, {'filename': '1_IM-0001-3001.dcm.png', 'projection': 'Lateral'}]
uid=2
  Reference images: [{'filename': '2_IM-0652-1001.dcm.png', 'projection': 'Frontal'}, {'filename': '2_IM-0652-2001.dcm.png', 'projection': 'Lateral'}]


In [9]:
import os
import csv
import shutil
import json
from pathlib import Path
from IPython.display import FileLink, display

# ── Paths ─────────────────────────────────────────────────────────────
metadata_path = 'output/metadata.json'
generated_images_dir = 'output/images'
original_images_root = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'

# ── Export directory structure ─────────────────────────────────────────
export_dir = 'verification_data'
synthetic_dir = os.path.join(export_dir, 'synthetic_images')
original_dir  = os.path.join(export_dir, 'original_images')

# Clean up any previous run
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)

os.makedirs(synthetic_dir, exist_ok=True)
os.makedirs(original_dir,  exist_ok=True)

if not os.path.exists(metadata_path):
    raise FileNotFoundError(f'Metadata file not found: {metadata_path}')

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f'Loaded metadata for {len(metadata)} report(s).\n')

synthetic_count = 0
original_count  = 0
mapping_rows    = []   # for the CSV comparison map

for item in metadata:
    uid = item.get('uid')
    if uid is None:
        continue

    # ── 1. Create a per-uid subfolder inside original_images ──────────
    uid_original_dir = os.path.join(original_dir, str(uid))
    os.makedirs(uid_original_dir, exist_ok=True)

    # ── 2. Copy original / reference images ──────────────────────────
    ref_filenames_copied = []
    for ref_img in item.get('reference_images', []):
        ref_filename   = ref_img.get('filename')
        ref_projection = ref_img.get('projection', '')
        if not ref_filename:
            continue
        src_ref = os.path.join(original_images_root, ref_filename)
        if os.path.exists(src_ref):
            shutil.copy2(src_ref, os.path.join(uid_original_dir, ref_filename))
            ref_filenames_copied.append(f'{ref_filename} ({ref_projection})')
            original_count += 1
        else:
            print(f'  ⚠ Original image not found for uid={uid}: {src_ref}')

    # ── 3. Copy generated / synthetic images ─────────────────────────
    #    The pipeline stores image_path inside each view entry.
    #    Fall back to simple {uid}.png if views data is unavailable.
    synth_filenames_copied = []
    views = item.get('views', [])

    if views:
        for view_entry in views:
            img_path = view_entry.get('image_path')
            if img_path and os.path.exists(img_path):
                dest_name = os.path.basename(img_path)
                shutil.copy2(img_path, os.path.join(synthetic_dir, dest_name))
                synth_filenames_copied.append(dest_name)
                synthetic_count += 1
    else:
        # Fallback: look for {uid}.png directly
        fallback_name = f'{uid}.png'
        fallback_src  = os.path.join(generated_images_dir, fallback_name)
        if os.path.exists(fallback_src):
            shutil.copy2(fallback_src, os.path.join(synthetic_dir, fallback_name))
            synth_filenames_copied.append(fallback_name)
            synthetic_count += 1

    # ── 4. Build mapping row ──────────────────────────────────────────
    mapping_rows.append({
        'uid': uid,
        'synthetic_images': '; '.join(synth_filenames_copied),
        'original_images':  '; '.join(ref_filenames_copied),
    })

    print(f'  ✓ uid={uid}  |  synthetic: {len(synth_filenames_copied)}  |  originals: {len(ref_filenames_copied)}')

# ── 5. Write comparison mapping CSV ───────────────────────────────────
mapping_csv_path = os.path.join(export_dir, 'image_mapping.csv')
with open(mapping_csv_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['uid', 'synthetic_images', 'original_images'])
    writer.writeheader()
    writer.writerows(mapping_rows)

print(f'\n── Summary ────────────────────────────────────────────────')
print(f'  Synthetic images copied : {synthetic_count}')
print(f'  Original images copied  : {original_count}')
print(f'  Mapping CSV             : {mapping_csv_path}')

# ── 6. Zip everything and provide download link ──────────────────────
zip_path = shutil.make_archive('verification_data', 'zip', '.', export_dir)
print(f'\n✅ Created {zip_path} for download.')
display(FileLink('verification_data.zip'))


Loaded metadata for 3 report(s).

  ✓ uid=1  |  synthetic: 2  |  originals: 2
  ✓ uid=2  |  synthetic: 2  |  originals: 2
  ✓ uid=3  |  synthetic: 2  |  originals: 2

── Summary ────────────────────────────────────────────────
  Synthetic images copied : 6
  Original images copied  : 6
  Mapping CSV             : verification_data/image_mapping.csv

✅ Created /kaggle/working/synth-dataset-pipeline/verification_data.zip for download.


/kaggle/working/synth-dataset-pipeline/verification_data.zip